# ImageNet-1k Baseline Pipeline

This notebook is separated from the CIFAR/HBCC workflow and is dedicated to paper-aligned ImageNet-1k baselines. It supports two paths:

1. Official Context-Cluster entrypoint from `docs/Context-Cluster/train.py`.
2. This repository training/benchmark loop using `configs/imagenet/*.yaml`.

ImageNet can be used either as a prepared `ImageFolder` layout (`IMAGENET_ROOT/train`, `IMAGENET_ROOT/val`) or directly from the Kaggle ImageNet Object Localization Challenge layout. Matching the Context-Cluster paper, training uses the official training split and reported accuracy uses the official validation split.

In [ ]:
from __future__ import annotations

import json
import subprocess
import sys
import time
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd()
if not (ROOT / "tools" / "train.py").exists():
    ROOT = ROOT.parent
assert (ROOT / "tools" / "train.py").exists(), f"Run this notebook from the repo root or notebooks dir. Got {ROOT}"

COC_PYTHON = Path(r"D:\Anaconda\envs\CoC\python.exe")
PYTHON = COC_PYTHON if COC_PYTHON.exists() else Path(sys.executable)

print("Repo:", ROOT)
print("Python:", PYTHON)


## Switches

Set the data mode first, then enable one or more phases. On Kaggle, `USE_DIRECT_KAGGLE_IMAGENET=True` avoids creating symlinks and reads the raw Kaggle layout directly.

In [ ]:
IMAGENET_ROOT = "data/imagenet"
KAGGLE_IMAGENET_SOURCE = "/kaggle/input/imagenet-object-localization-challenge"

USE_DIRECT_KAGGLE_IMAGENET = Path(KAGGLE_IMAGENET_SOURCE).exists()
PREPARE_KAGGLE_IMAGENET = False
RUN_ENV_CHECKS = True
RUN_REPO_IMAGENET_BASELINES = False
RUN_OFFICIAL_COC_ENTRYPOINT = False
RUN_IMAGENET_BENCHMARKS = False
RUN_KAGGLE_TEST_SUBMISSION = False
RUN_SUMMARY = True

SUBMISSION_CONFIG = "configs/imagenet/hbcc_accuracy_medium_imagenet.yaml"
SUBMISSION_CHECKPOINT = "runs_imagenet/hbcc_accuracy_medium_imagenet/best.pth"
SUBMISSION_OUTPUT = "/kaggle/working/submission.csv" if Path("/kaggle/working").exists() else "submission.csv"
SUBMISSION_TOPK = 5
SUBMISSION_BATCH_SIZE = 128
SUBMISSION_WORKERS = 2

BENCHMARK_BATCHES = [1, 16, 64, 128]
BENCHMARK_WARMUP = 30
BENCHMARK_RUNS = 100

# Debug settings for quick smoke benchmark only.
DEBUG_BENCHMARK_BATCHES = [1]
DEBUG_BENCHMARK_WARMUP = 1
DEBUG_BENCHMARK_RUNS = 1


In [ ]:
def _looks_like_tqdm(line: str) -> bool:
    text = line.strip()
    return "%|" in text or text.startswith("val:") or text.startswith("test:") or text.startswith("train ")


def run_py(args: list[str], check: bool = True) -> int:
    cmd = [str(PYTHON), *args]
    print("\n$", " ".join(cmd))
    start = time.perf_counter()
    process = subprocess.Popen(
        cmd,
        cwd=ROOT,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        bufsize=1,
    )
    assert process.stdout is not None
    progress_handle = display(Markdown(""), display_id=True)
    last_progress = ""
    for line in process.stdout:
        clean = line.rstrip("\r\n")
        if _looks_like_tqdm(clean):
            last_progress = clean
            progress_handle.update(Markdown(f"```text\n{last_progress}\n```"))
        else:
            print(line, end="")
    code = process.wait()
    elapsed = time.perf_counter() - start
    if last_progress:
        progress_handle.update(Markdown(f"```text\n{last_progress}\n```"))
    print(f"\n[exit={code}] elapsed={elapsed:.1f}s")
    if check and code != 0:
        raise RuntimeError(f"Command failed with exit code {code}: {' '.join(cmd)}")
    return code


def override_args(overrides: list[str] | None) -> list[str]:
    args: list[str] = []
    for item in overrides or []:
        args.extend(["--override", item])
    return args


def train(config: str, output: str, overrides: list[str] | None = None, extra: list[str] | None = None) -> None:
    args = ["tools/train.py", "--config", config, "--output", output]
    args += override_args(overrides or [])
    args += extra or []
    run_py(args)


def benchmark(config: str, checkpoint: str | None = None, debug: bool = False, profile: bool = True) -> None:
    batches = DEBUG_BENCHMARK_BATCHES if debug else BENCHMARK_BATCHES
    warmup = DEBUG_BENCHMARK_WARMUP if debug else BENCHMARK_WARMUP
    runs = DEBUG_BENCHMARK_RUNS if debug else BENCHMARK_RUNS
    args = [
        "tools/benchmark.py",
        "--config", config,
        "--output", "results/benchmark_imagenet",
        "--batch-sizes", *[str(v) for v in batches],
        "--warmup", str(warmup),
        "--runs", str(runs),
    ]
    if checkpoint and (ROOT / checkpoint).exists():
        args += ["--checkpoint", checkpoint]
    elif checkpoint:
        print(f"Skip checkpoint for {config}; not found: {checkpoint}")
    if profile:
        args.append("--profile")
    run_py(args)


def use_direct_kaggle_layout() -> bool:
    return bool(USE_DIRECT_KAGGLE_IMAGENET) and Path(KAGGLE_IMAGENET_SOURCE).exists()


def imagenet_root_path() -> Path:
    root = Path(KAGGLE_IMAGENET_SOURCE) if use_direct_kaggle_layout() else Path(IMAGENET_ROOT)
    return root if root.is_absolute() else ROOT / root


def imagenet_train_path() -> Path:
    root = imagenet_root_path()
    return root / "ILSVRC" / "Data" / "CLS-LOC" / "train" if use_direct_kaggle_layout() else root / "train"


def imagenet_val_path() -> Path:
    root = imagenet_root_path()
    return root / "ILSVRC" / "Data" / "CLS-LOC" / "val" if use_direct_kaggle_layout() else root / "val"


def imagenet_data_overrides() -> list[str]:
    if use_direct_kaggle_layout():
        return [
            f"data.root={KAGGLE_IMAGENET_SOURCE}",
            "data.layout=kaggle_cls_loc",
            "data.train_split=ILSVRC/Data/CLS-LOC/train",
            "data.val_split=ILSVRC/Data/CLS-LOC/val",
        ]
    return [f"data.root={IMAGENET_ROOT}"]


## Kaggle Preparation And Environment Check

In [ ]:
if PREPARE_KAGGLE_IMAGENET and use_direct_kaggle_layout():
    print("Direct Kaggle layout is enabled; skip symlink preparation. Set USE_DIRECT_KAGGLE_IMAGENET=False to prepare ImageFolder layout.")
elif PREPARE_KAGGLE_IMAGENET:
    run_py([
        "scripts/prepare_kaggle_imagenet.py",
        "--source", KAGGLE_IMAGENET_SOURCE,
        "--output", IMAGENET_ROOT,
        "--mode", "symlink",
    ])

if RUN_ENV_CHECKS:
    run_py(["-c", "import timm; print(timm.__version__)"])
    official_code_candidates = [ROOT / "docs" / "Context-Cluster" / "models" / "context_cluster.py", ROOT / "docs" / "context-cluster" / "models" / "context_cluster.py"]
    official_code = next((path for path in official_code_candidates if path.exists()), official_code_candidates[0])
    if official_code.exists():
        run_py(["-c", "from lightweight_hbcc.config import load_config; from lightweight_hbcc.models import build_model; cfg=load_config('configs/imagenet/official_coc_tiny_imagenet.yaml'); m=build_model(cfg); print('official_coc_tiny_imagenet', getattr(m, 'num_classes', None), sum(p.numel() for p in m.parameters()))"])
    else:
        print("Skip official CoC check; missing:", official_code)
    for cfg_path in ["configs/imagenet/hbcc_accuracy_small_imagenet.yaml", "configs/imagenet/hbcc_accuracy_medium_imagenet.yaml"]:
        run_py(["-c", f"from lightweight_hbcc.config import load_config; from lightweight_hbcc.models import build_model; cfg=load_config('{cfg_path}'); m=build_model(cfg); print('{cfg_path}', getattr(m, 'num_classes', None), sum(p.numel() for p in m.parameters()))"])
    root = imagenet_root_path()
    print("ImageNet root:", root)
    print("direct Kaggle layout:", use_direct_kaggle_layout())
    print("train exists:", imagenet_train_path().exists(), imagenet_train_path())
    print("val exists:", imagenet_val_path().exists(), imagenet_val_path())


## Repo Trainer Baselines And HBCC

These jobs use this repository training loop with the selected ImageNet HBCC accuracy candidates converted from the CIFAR-10 recipes.

In [ ]:
IMAGENET_BASELINE_CONFIGS = [
    "configs/imagenet/hbcc_accuracy_small_imagenet.yaml",
    "configs/imagenet/hbcc_accuracy_medium_imagenet.yaml",
]

if RUN_REPO_IMAGENET_BASELINES:
    if not imagenet_train_path().exists() or not imagenet_val_path().exists():
        raise FileNotFoundError(f"ImageNet train/val folders are not ready: {imagenet_train_path()} / {imagenet_val_path()}")
    for cfg in IMAGENET_BASELINE_CONFIGS:
        train(cfg, "runs_imagenet", overrides=imagenet_data_overrides())


## Official Context-Cluster Entrypoint

This path calls `docs/Context-Cluster/train.py` directly, matching the official code path.

In [ ]:
OFFICIAL_COC_RUN = "runs_imagenet_official/official_coc_tiny_imagenet"
OFFICIAL_COC_CHECKPOINT = f"{OFFICIAL_COC_RUN}/model_best.pth.tar"

if RUN_OFFICIAL_COC_ENTRYPOINT:
    if use_direct_kaggle_layout():
        raise RuntimeError("Official Context-Cluster entrypoint cannot read Kaggle flat val directly. Set USE_DIRECT_KAGGLE_IMAGENET=False and PREPARE_KAGGLE_IMAGENET=True first.")
    if not (ROOT / IMAGENET_ROOT / "train").exists() or not (ROOT / IMAGENET_ROOT / "val").exists():
        raise FileNotFoundError(f"Prepare ImageNet at {ROOT / IMAGENET_ROOT} with train/ and val/ folders")
    official_train = ROOT / "docs" / "Context-Cluster" / "train.py"
    official_validate = ROOT / "docs" / "Context-Cluster" / "validate.py"
    if not official_train.exists():
        raise FileNotFoundError("Official Context-Cluster code is missing: docs/Context-Cluster/train.py")
    run_py([
        str(official_train.relative_to(ROOT)),
        "--data_dir", IMAGENET_ROOT,
        "--train-split", "train",
        "--val-split", "val",
        "--model", "coc_tiny",
        "--num-classes", "1000",
        "--batch-size", "128",
        "--validation-batch-size", "128",
        "--lr", "0.001",
        "--drop-path", "0.1",
        "--amp",
        "--output", "runs_imagenet_official",
        "--experiment", "official_coc_tiny_imagenet",
    ])
    if (ROOT / OFFICIAL_COC_CHECKPOINT).exists():
        run_py([
            str(official_validate.relative_to(ROOT)), IMAGENET_ROOT,
            "--split", "val",
            "--model", "coc_tiny",
            "--num-classes", "1000",
            "--batch-size", "128",
            "--checkpoint", OFFICIAL_COC_CHECKPOINT,
            "--amp",
        ])


## Benchmark

In [ ]:
IMAGENET_BENCHMARK_JOBS = [
    ("configs/imagenet/hbcc_accuracy_small_imagenet.yaml", "runs_imagenet/hbcc_accuracy_small_imagenet/best.pth"),
    ("configs/imagenet/hbcc_accuracy_medium_imagenet.yaml", "runs_imagenet/hbcc_accuracy_medium_imagenet/best.pth"),
]

if RUN_IMAGENET_BENCHMARKS:
    for cfg, ckpt in IMAGENET_BENCHMARK_JOBS:
        benchmark(cfg, checkpoint=ckpt, debug=False, profile=True)


## Kaggle Test Submission

This creates `ImageId,PredictionString` for late submission on the ImageNet Object Localization Challenge test split. The current models are classifiers, so each predicted class uses the full image as its bounding box.

In [ ]:
if RUN_KAGGLE_TEST_SUBMISSION:
    import csv
    from PIL import Image
    import torch
    from torch.utils.data import DataLoader, Dataset
    from tqdm.auto import tqdm

    from lightweight_hbcc.config import load_config
    from lightweight_hbcc.data import _transforms
    from lightweight_hbcc.engine import load_checkpoint, resolve_device
    from lightweight_hbcc.models import build_model

    IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

    def _first_existing(paths):
        for path in paths:
            path = Path(path)
            if path.exists():
                return path
        return None

    def _find_sample_submission(source_root: Path) -> Path | None:
        known = [
            source_root / "LOC_sample_submission.csv",
            source_root / "ILSVRC" / "LOC_sample_submission.csv",
        ]
        found = _first_existing(known)
        if found is not None:
            return found
        matches = sorted(source_root.rglob("LOC_sample_submission.csv")) if source_root.exists() else []
        return matches[0] if matches else None

    def _find_test_dir(source_root: Path) -> Path:
        candidates = [
            source_root / "ILSVRC" / "Data" / "CLS-LOC" / "test",
            source_root / "Data" / "CLS-LOC" / "test",
            source_root / "test",
            ROOT / IMAGENET_ROOT / "test",
        ]
        found = _first_existing(candidates)
        if found is None:
            raise FileNotFoundError(f"Could not find ImageNet test folder under {source_root}")
        return found

    def _index_images(image_dir: Path) -> dict[str, Path]:
        files = [p for p in image_dir.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_SUFFIXES]
        if not files:
            files = [p for p in image_dir.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_SUFFIXES]
        return {p.stem: p for p in files}

    class KaggleImageNetTestDataset(Dataset):
        def __init__(self, image_ids: list[str], image_dir: Path, transform) -> None:
            self.image_ids = image_ids
            self.transform = transform
            self.files_by_stem = _index_images(image_dir)
            missing = [image_id for image_id in image_ids if image_id not in self.files_by_stem]
            if missing:
                preview = ", ".join(missing[:5])
                raise FileNotFoundError(f"Missing {len(missing)} test images. First missing: {preview}")

        def __len__(self) -> int:
            return len(self.image_ids)

        def __getitem__(self, index: int):
            image_id = self.image_ids[index]
            path = self.files_by_stem[image_id]
            with Image.open(path) as image:
                image = image.convert("RGB")
                width, height = image.size
                tensor = self.transform(image)
            return tensor, image_id, width, height

    source_root = Path(KAGGLE_IMAGENET_SOURCE)
    sample_path = _find_sample_submission(source_root)
    test_dir = _find_test_dir(source_root)
    if sample_path is not None:
        sample_df = pd.read_csv(sample_path)
        image_ids = sample_df["ImageId"].astype(str).tolist()
        print("sample:", sample_path)
    else:
        image_ids = sorted(_index_images(test_dir))
        print("No LOC_sample_submission.csv found; using sorted test image stems.")
    print("test images:", len(image_ids), "from", test_dir)

    train_class_dir = imagenet_train_path()
    index_to_wnid = [p.name for p in sorted(train_class_dir.iterdir()) if p.is_dir()]
    if len(index_to_wnid) != 1000:
        raise RuntimeError(f"Expected 1000 ImageNet class folders, found {len(index_to_wnid)} in {train_class_dir}")

    cfg = load_config(ROOT / SUBMISSION_CONFIG)
    transform = _transforms(str(cfg.get("data", {}).get("name", "imagenet1k")), False, False, cfg.get("data", {}))
    device = resolve_device("auto")
    model = build_model(cfg).to(device).eval()
    checkpoint = ROOT / SUBMISSION_CHECKPOINT
    if not checkpoint.exists():
        raise FileNotFoundError(checkpoint)
    load_checkpoint(model, checkpoint, device, strict=True)

    dataset = KaggleImageNetTestDataset(image_ids, test_dir, transform)
    loader = DataLoader(dataset, batch_size=SUBMISSION_BATCH_SIZE, shuffle=False, num_workers=SUBMISSION_WORKERS, pin_memory=(device.type == "cuda"))

    output_path = Path(SUBMISSION_OUTPUT)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    topk = min(int(SUBMISSION_TOPK), len(index_to_wnid))
    rows_written = 0
    with output_path.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=["ImageId", "PredictionString"])
        writer.writeheader()
        with torch.no_grad():
            for images, batch_ids, widths, heights in tqdm(loader, desc="predict test"):
                logits = model(images.to(device, non_blocking=True))
                pred_indices = logits.topk(topk, dim=1).indices.cpu().tolist()
                for image_id, width, height, preds in zip(batch_ids, widths.tolist(), heights.tolist(), pred_indices):
                    xmax = max(int(width) - 1, 0)
                    ymax = max(int(height) - 1, 0)
                    parts = []
                    for class_index in preds:
                        parts.extend([index_to_wnid[int(class_index)], "0", "0", str(xmax), str(ymax)])
                    writer.writerow({"ImageId": image_id, "PredictionString": " ".join(parts)})
                    rows_written += 1

    print("submission:", output_path)
    print("rows:", rows_written)
    pd.read_csv(output_path).head()


## Summary

In [ ]:
def read_jsonl(path: Path) -> list[dict]:
    if not path.exists():
        return []
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]


def collect_training_summary(root_pattern: str = "runs_imagenet*/**/metrics.jsonl") -> pd.DataFrame:
    rows = []
    for metrics_path in ROOT.glob(root_pattern):
        records = read_jsonl(metrics_path)
        val_records = [r for r in records if "val_acc1" in r]
        if not val_records:
            continue
        best = max(val_records, key=lambda r: r["val_acc1"])
        rows.append({
            "run_dir": str(metrics_path.parent.relative_to(ROOT)),
            "experiment": metrics_path.parent.name,
            "best_epoch": best.get("epoch"),
            "reported_val_acc1": best.get("val_acc1"),
            "reported_val_acc5": best.get("val_acc5"),
        })
    return pd.DataFrame(rows).sort_values("reported_val_acc1", ascending=False) if rows else pd.DataFrame()


summary_df = collect_training_summary() if RUN_SUMMARY else pd.DataFrame()
summary_df
